# NRC-Cal: Full Experiment

Evaluate BASE, QR/QRC/QRTC/QREGC adapters, and NRC-Cal across QRT-57.

**Attribution.** Existing theory: published NRC equations. Proposed method: NRC-Cal; all baselines retain upstream attribution.

**Resources.** Expected runtime varies with dataset/checkpoint availability; GPU memory is limited to one frozen model and one batch. Expected output paths are printed by executable cells. Every code cell independently resolves Drive/GitHub/SSH-synchronized project source before it acts.


In [ ]:
# Self-contained Colab bootstrap: Drive first, then configured GitHub/SSH roots.
from pathlib import Path
import importlib.util
import os, sys

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

candidates = []
if os.environ.get("NRC_CAL_PROJECT_ROOT"):
    candidates.append(Path(os.environ["NRC_CAL_PROJECT_ROOT"]))
candidates += [Path("/content/drive/MyDrive/NRC-Cal/project"), Path("/content/NRC-Cal/project"), Path.cwd().parent, Path.cwd()]
PROJECT_ROOT = next((p.resolve() for p in candidates if (p / "src").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("NRC-Cal project unavailable. Run 00_environment.ipynb or set NRC_CAL_PROJECT_ROOT after Drive, GitHub, or SSH/rsync synchronization.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
runtime_path = PROJECT_ROOT / "src" / "utils" / "runtime.py"
if not runtime_path.is_file():
    raise FileNotFoundError(f"Missing runtime helper: {runtime_path}")
spec = importlib.util.spec_from_file_location("nrc_cal_runtime", runtime_path)
if spec is None or spec.loader is None:
    raise ImportError(f"Unable to load runtime helper from {runtime_path}")
runtime_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runtime_module)
configure_runtime = runtime_module.configure_runtime
gpu_summary = runtime_module.gpu_summary
RUNTIME = configure_runtime(PROJECT_ROOT)
print(f"PROJECT_ROOT={RUNTIME.project_root}")
print(f"CHECKPOINT_ROOT={RUNTIME.checkpoint_root}")
print(f"CUDA available={RUNTIME.cuda_available}; PyTorch CUDA={RUNTIME.cuda_version}; GPU={RUNTIME.gpu_name}")
print(f"RAM={RUNTIME.ram_gib:.2f} GiB; driver={gpu_summary()}")


## Equations and attribution

**Published NRC theory (NeurIPS 2024).** `NRC1=M^-1 sum_i ||h~_i-P_H_PCA_n(h~_i)||_2^2`; `NRC2=M^-1 sum_i ||h~_i-P_W^T(h~_i)||_2^2`; and `NRC3=||WW^T/||WW^T||_F-(Sigma^(1/2)-gamma^(1/2)I_n)/||Sigma^(1/2)-gamma^(1/2)I_n||_F||_F^2`, where `gamma in (0,lambda_min)`. These are implemented exactly in `geometry.nrc`. **NRC-Cal proposal.** `D_i=w1 r1_i+w2 r2_i+w3 NRC3`, `D_dataset=M^-1 sum_i D_i`, and the frozen scale map are new equations, specified with assumptions and proof sketch in `docs/methodology.md`. No proposed equation is attributed to the NRC papers.


In [ ]:
import pandas as pd
from calibration.baselines import baseline_registry
from metrics.experiment import run_frozen_nrc_cal
print("Methods:", baseline_registry())
artifact_root = PROJECT_ROOT / "outputs" / "artifacts"
rows = []
for dataset_dir in sorted(path for path in artifact_root.iterdir() if path.is_dir()) if artifact_root.exists() else []:
    required = [dataset_dir / name for name in ("calibration_features.npz", "calibration_predictions.npz", "test_features.npz", "test_predictions.npz", "mean_head_weight.npy")]
    if not all(path.exists() for path in required):
        print("Skipping incomplete artifact set:", dataset_dir); continue
    rows.append(run_frozen_nrc_cal(dataset_dir.name, "frozen", *required[:4], mean_head_weight=__import__("numpy").load(required[4]), output_path=dataset_dir / "nrc_cal_results.csv"))
if rows:
    pd.concat(rows, ignore_index=True).to_csv(PROJECT_ROOT / "outputs" / "results" / "full_experiment.csv", index=False)
    print("Completed frozen BASE/NRC-Cal runs:", len(rows))
else:
    print("No complete artifact sets yet. Notebook 04 creates the required immutable artifact contract for each available checkpoint.")


## Reproducibility note

This notebook writes only under `outputs/`, `figures/`, or the Drive-backed checkpoint root. It does not retrain a model. Preserve the environment JSON from notebook 00 and the upstream checkpoint inventory when exporting results.
